In [0]:
df = spark.table("clean_transactions")

from pyspark.ml.feature import VectorAssembler
feature_cols = [c for c in df.columns if c not in ["Class"]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
model_data = assembler.transform(df)

train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="Class", numTrees=100, seed=42)
rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="Class", predictionCol="prediction")

acc = evaluator.evaluate(rf_predictions, {evaluator.metricName: "accuracy"})
prec = evaluator.evaluate(rf_predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
rec = evaluator.evaluate(rf_predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})
print(f"Random Forest — Accuracy: {acc:.4f}, Fraud Precision: {prec:.4f}, Fraud Recall: {rec:.4f}")

In [0]:
import matplotlib.pyplot as plt

# Class imbalance chart
class_counts = df.groupBy("Class").count().toPandas()
plt.figure(figsize=(6,4))
plt.bar(class_counts["Class"].astype(str), class_counts["count"])
plt.title("Class Distribution: Legitimate vs Fraud")
plt.xlabel("Class (0=Legit, 1=Fraud)")
plt.ylabel("Count")
plt.yscale("log")
plt.savefig("/tmp/class_distribution.png")
plt.show()

In [0]:
# Confusion matrix for your best model (swap rf_predictions for whichever wins)
import pandas as pd
import numpy as np

# Create confusion matrix using DataFrame operations
preds_df = rf_predictions.select("prediction", "Class").toPandas()
cm = pd.crosstab(preds_df["Class"], preds_df["prediction"], rownames=["Actual"], colnames=["Predicted"]).values

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
for i in range(2):
    for j in range(2):
        plt.text(j, i, int(cm[i,j]), ha="center", va="center")
plt.savefig("/tmp/confusion_matrix.png")
plt.show()